In [10]:
print("Hello, World!")

Hello, World!


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

c:\Users\SELVAGANAPATHI\anaconda3\envs\tune\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # NF4 (important)
    bnb_4bit_use_double_quant=True,  # Double quantization
    bnb_4bit_compute_dtype=torch.float16
)

In [3]:
import accelerate
print(accelerate.__version__)

1.13.0


In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv("HF_TOKEN")
if token:
    login(token)
else:
    print("HF_TOKEN not found in .env file")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `gemma tune` has been saved to C:\Users\SELVAGANAPATHI\.cache\huggingface\stored_tokens
Your token has been saved to C:\Users\SELVAGANAPATHI\.cache\huggingface\token
Login successful.
The current active token is: `gemma tune`


In [7]:
from huggingface_hub import logging
logging.set_verbosity_info()

In [ ]:
model_name = "LiquidAI/LFM2.5-350M"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

c:\Users\SELVAGANAPATHI\anaconda3\envs\tune\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SELVAGANAPATHI\.cache\huggingface\hub\models--LiquidAI--LFM2.5-350M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:01<00:00, 84.42it/s] 


In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [12]:
inputs = tokenizer("What is the capital of France?", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

What is the capital of France?

A) Paris

B) Rome

C) London

D) Berlin

Answer: A


In [13]:
print(outputs)

tensor([[    1,  3493,   856,   779,  5706,   803,  4481,   540,   509,   542,
           518,  5242,   509,   543,   518, 10000,   509,   544,   518,  5096,
           509,   545,   518,  7053,   509, 40131,   535,   835,     7]],
       device='cuda:0')


In [ ]:
model_name = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True
)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token